# Bu Dersi Google Colab'da Çalıştır

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BILSEM-BT/Python/blob/main/23-PythonSiniflandirmaAlgoritmalari.ipynb)

Bu notebook GitHub üzerinde ders dokümanı olarak yayımlanır. Kodları çalıştırmak ve üzerinde denemeler yapmak için yukarıdaki **Open in Colab** butonunu kullanabilirsiniz.

### Nasıl çalışacağız?

1. **Open in Colab** butonuna tıklayın.
2. Açılan notebook'taki kod hücrelerini `▶` düğmesiyle çalıştırın.
3. Kodları değiştirerek farklı sonuçları deneyin.
4. Çalışmalarınız kendi Colab çalışma alanınızda tutulur; bu GitHub'daki ana ders dosyasını değiştirmez.

> **Önemli:** GitHub'daki bu dosya dersin ana ve değiştirilmeyen kaynağıdır. Colab'da yaptığınız değişiklikler bu dosyaya otomatik olarak yazılmaz.

---

# 23 - Python'da Sınıflandırma Algoritmaları ve Model Karşılaştırma

**Niyazi Sayın BİLSEM**  
**Bilişim Teknolojileri Dersi**  
**Ders Öğretmeni: Ersin ŞANLI**

Bir önceki yapay zeka dersinde ilk sınıflandırma modelimizi oluşturduk. Bu derste aynı problemi farklı algoritmalarla çözecek ve hangi modelin hangi durumda daha uygun olabileceğini inceleyeceğiz.

Kullanacağımız başlıca modeller:

- Dummy Classifier
- Logistic Regression
- K-Nearest Neighbors
- Decision Tree
- Random Forest
- Support Vector Machine
- Gaussian Naive Bayes

Bu dersin sonunda öğrencinin:

- sınıflandırma problemini tanımlayabilmesi,
- baseline model kullanabilmesi,
- farklı sınıflandırma algoritmalarını eğitebilmesi,
- ölçeklendirme gerektiren modelleri ayırt edebilmesi,
- Pipeline kullanabilmesi,
- accuracy, precision, recall ve F1-score hesaplayabilmesi,
- confusion matrix okuyabilmesi,
- model sonuçlarını aynı tabloda karşılaştırabilmesi,
- yeni veri için sınıf ve olasılık tahmini üretebilmesi,
- model seçiminin yalnızca tek bir başarı değerine dayanmadığını anlayabilmesi

hedeflenmektedir.


# 1. Sınıflandırma Problemi

Sınıflandırma problemlerinde hedef değişken kategorik bir sınıftır.

Örnekler:

```text
E-posta → Spam / Normal
İşlem → Şüpheli / Normal
Sensör Durumu → Arızalı / Sağlıklı
Öğrenci Çalışması → Tamamlandı / Tamamlanmadı
Bitki → Tür 1 / Tür 2 / Tür 3
```

Bu derste iki sınıflı sentetik bir veri kümesi kullanacağız.


# 2. Neden Sentetik Veri Kullanıyoruz?

Bu dersin amacı algoritmaları karşılaştırmaktır.

İnternet bağlantısı veya dış veri dosyasına bağımlı olmadan her öğrencinin aynı sonucu alabilmesi için scikit-learn ile kontrollü bir eğitim veri kümesi oluşturacağız.

Veri kümesindeki değişkenleri örnek bir **teknik sistem durumu** gibi düşüneceğiz:

```text
0 → Normal
1 → İncelenmeli
```

Bu sınıflar yalnızca eğitim amacıyla kullanılmaktadır.


# 3. Gerekli Kütüphaneler

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split


# 4. Veri Kümesini Oluşturmak

In [ ]:
X_array, y_array = make_classification(
    n_samples=600,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_repeated=0,
    n_classes=2,
    weights=[0.65, 0.35],
    class_sep=1.15,
    flip_y=0.03,
    random_state=42
)

ozellik_adlari = [
    "SensorA",
    "SensorB",
    "SensorC",
    "SensorD",
    "SensorE",
    "SensorF",
    "SensorG",
    "SensorH"
]

X = pd.DataFrame(
    X_array,
    columns=ozellik_adlari
)

y = pd.Series(
    y_array,
    name="Durum"
)

X.head()


# 5. Veri Kümesinin Boyutu

In [ ]:
print("Özellik tablosu:", X.shape)
print("Hedef:", y.shape)


# 6. İlk Satırları Birlikte Görmek

In [ ]:
veri = X.copy()
veri["Durum"] = y

veri.head()


# 7. Veri Türleri

In [ ]:
print(veri.dtypes)


# 8. Eksik Veri Kontrolü

In [ ]:
print(veri.isna().sum())


# 9. Sınıf Dağılımı

In [ ]:
sinif_sayilari = y.value_counts().sort_index()

print(sinif_sayilari)


Bu veri kümesinde sınıflar tamamen eşit değildir.

Bu durum bize yalnızca accuracy değerine bakmanın neden bazı problemlerde yetersiz olabileceğini gösterecektir.


# 10. Sınıf Oranları

In [ ]:
sinif_oranlari = (
    y.value_counts(
        normalize=True
    )
    .sort_index()
    * 100
)

print(sinif_oranlari)


# 11. Sınıf Dağılım Grafiği

In [ ]:
plt.bar(
    ["Normal", "İncelenmeli"],
    sinif_sayilari.values
)

plt.ylabel("Örnek Sayısı")
plt.title("Sınıf Dağılımı")
plt.show()


# 12. Temel İstatistikler

In [ ]:
X.describe()


# 13. İki Özelliği Görselleştirmek

In [ ]:
for sinif in [0, 1]:
    secim = y == sinif

    plt.scatter(
        X.loc[secim, "SensorA"],
        X.loc[secim, "SensorB"],
        label="Normal" if sinif == 0 else "İncelenmeli"
    )

plt.xlabel("SensorA")
plt.ylabel("SensorB")
plt.title("İki Özellik Üzerinde Sınıflar")
plt.legend()
plt.show()


Gerçek problemlerimiz sekiz özelliklidir. İki boyutlu grafik, yalnızca verinin küçük bir bölümünü görmemizi sağlar.

Makine öğrenmesi modeli sekiz özelliğin tamamını birlikte kullanabilir.


# 14. Train-Test Ayrımı

Modeli daha önce görmediği veriler üzerinde değerlendirmek için veriyi eğitim ve test olarak ayıracağız.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Eğitim:", X_train.shape)
print("Test:", X_test.shape)


`stratify=y` sınıf oranlarının eğitim ve test gruplarında korunmasına yardımcı olur.


# 15. Eğitim Sınıf Dağılımı

In [ ]:
print(
    y_train.value_counts(
        normalize=True
    ).sort_index()
)


# 16. Test Sınıf Dağılımı

In [ ]:
print(
    y_test.value_counts(
        normalize=True
    ).sort_index()
)


# 17. Baseline Model Nedir?

Yeni bir algoritmanın gerçekten yararlı olup olmadığını anlamak için çok basit bir başlangıç modeline ihtiyaç duyarız.

Bu modele **baseline** diyebiliriz.

Sınıflandırmada basit bir yaklaşım:

```text
Her zaman en sık görülen sınıfı tahmin et.
```

olabilir.


# 18. Dummy Classifier

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline.fit(
    X_train,
    y_train
)

baseline_pred = baseline.predict(
    X_test
)

print(
    "Baseline Accuracy:",
    baseline.score(
        X_test,
        y_test
    )
)


Model karşılaştırmalarımızda gerçek algoritmaların bu basit yaklaşımdan daha anlamlı sonuç üretmesini bekleriz.


# 19. Değerlendirme Metriklerini İçe Aktarmak

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# 20. Accuracy

Accuracy:

```text
Doğru Tahmin Sayısı
-------------------
Toplam Tahmin Sayısı
```

olarak düşünülebilir.


In [ ]:
baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred
)

print(
    "Baseline Accuracy:",
    baseline_accuracy
)


# 21. Precision

Precision, modelin pozitif sınıf olarak tahmin ettiklerinin ne kadarının gerçekten pozitif olduğunu inceler.

Basitleştirilmiş soru:

```text
Model "İncelenmeli" dediğinde ne kadar güvenilir?
```


In [ ]:
baseline_precision = precision_score(
    y_test,
    baseline_pred,
    zero_division=0
)

print(
    "Baseline Precision:",
    baseline_precision
)


# 22. Recall

Recall, gerçek pozitif örneklerin ne kadarını yakalayabildiğimizi inceler.

Basitleştirilmiş soru:

```text
Gerçek "İncelenmeli" örneklerin ne kadarını bulduk?
```


In [ ]:
baseline_recall = recall_score(
    y_test,
    baseline_pred,
    zero_division=0
)

print(
    "Baseline Recall:",
    baseline_recall
)


# 23. F1-Score

F1-score precision ve recall değerlerini birlikte değerlendiren bir ölçüdür.

Precision ve recall arasında denge gerektiğinde yararlı olabilir.


In [ ]:
baseline_f1 = f1_score(
    y_test,
    baseline_pred,
    zero_division=0
)

print(
    "Baseline F1:",
    baseline_f1
)


# 24. Confusion Matrix

İkili sınıflandırmada confusion matrix şu dört temel durumu gösterir:

```text
TN → Gerçek 0, Tahmin 0
FP → Gerçek 0, Tahmin 1
FN → Gerçek 1, Tahmin 0
TP → Gerçek 1, Tahmin 1
```


In [ ]:
baseline_cm = confusion_matrix(
    y_test,
    baseline_pred
)

print(baseline_cm)


# 25. Confusion Matrix Değerlerini Ayırmak

In [ ]:
tn, fp, fn, tp = baseline_cm.ravel()

print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)


# 26. İlk Gerçek Model: Logistic Regression

Adında regresyon geçmesine rağmen `LogisticRegression`, sınıflandırma için kullanılan bir modeldir.

Özellikle ikili sınıflandırmada güçlü ve açıklanabilir bir başlangıç modeli olabilir.


# 27. StandardScaler ve Pipeline

Logistic Regression, KNN ve SVM gibi modellerde özellik ölçeklerinin farklı olması önemli olabilir.

Ön işleme ve modeli Pipeline içinde birleştireceğiz.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


# 28. Logistic Regression Pipeline

In [ ]:
lojistik = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

lojistik.fit(
    X_train,
    y_train
)

lojistik_pred = lojistik.predict(
    X_test
)

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        lojistik_pred
    )
)


# 29. Logistic Regression Metrikleri

In [ ]:
print(
    "Precision:",
    precision_score(
        y_test,
        lojistik_pred
    )
)

print(
    "Recall:",
    recall_score(
        y_test,
        lojistik_pred
    )
)

print(
    "F1:",
    f1_score(
        y_test,
        lojistik_pred
    )
)


# 30. Logistic Regression Classification Report

In [ ]:
print(
    classification_report(
        y_test,
        lojistik_pred,
        target_names=[
            "Normal",
            "İncelenmeli"
        ]
    )
)


# 31. Logistic Regression Confusion Matrix

In [ ]:
lojistik_cm = confusion_matrix(
    y_test,
    lojistik_pred
)

print(lojistik_cm)


# 32. K-Nearest Neighbors

KNN yeni örneği en yakın komşularına bakarak sınıflandırır.

Bu modelde özellik ölçeklendirme özellikle önemlidir.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        KNeighborsClassifier(
            n_neighbors=7
        )
    )
])

knn.fit(
    X_train,
    y_train
)

knn_pred = knn.predict(
    X_test
)

print(
    "KNN Accuracy:",
    accuracy_score(
        y_test,
        knn_pred
    )
)


# 33. KNN'de K Değerinin Etkisi

In [ ]:
knn_sonuclari = []

for k in range(1, 22, 2):
    deneme = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            KNeighborsClassifier(
                n_neighbors=k
            )
        )
    ])

    deneme.fit(
        X_train,
        y_train
    )

    tahmin = deneme.predict(
        X_test
    )

    knn_sonuclari.append({
        "K": k,
        "Accuracy": accuracy_score(
            y_test,
            tahmin
        ),
        "F1": f1_score(
            y_test,
            tahmin
        )
    })

knn_df = pd.DataFrame(
    knn_sonuclari
)

knn_df


# 34. K Değerlerini Grafikle Karşılaştırmak

In [ ]:
plt.plot(
    knn_df["K"],
    knn_df["Accuracy"],
    marker="o",
    label="Accuracy"
)

plt.plot(
    knn_df["K"],
    knn_df["F1"],
    marker="o",
    label="F1"
)

plt.xlabel("K")
plt.ylabel("Skor")
plt.title("KNN Hiperparametre Karşılaştırması")
plt.legend()
plt.grid()
plt.show()


Bu derste K değerlerini test setine bakarak yalnızca kavramı göstermek için karşılaştırıyoruz.

Gerçek model seçimi için test setini hiperparametre ayarında kullanmak yerine cross-validation yaklaşımı tercih edilir. Bunu bir sonraki derste ayrıntılı uygulayacağız.


# 35. Decision Tree

Decision Tree, veriyi art arda koşullarla bölerek karar kuralları oluşturur.

Örnek düşünce:

```text
SensorC < değer?
↓
Evet / Hayır
↓
Yeni koşul
↓
Sınıf
```

Karar ağaçlarının önemli avantajlarından biri yorumlanabilir olmalarıdır.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

agac = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=5,
    random_state=42
)

agac.fit(
    X_train,
    y_train
)

agac_pred = agac.predict(
    X_test
)

print(
    "Decision Tree Accuracy:",
    accuracy_score(
        y_test,
        agac_pred
    )
)


# 36. Karar Ağacı Derinliği

Ağaç çok derinleşirse eğitim verisine aşırı uyum sağlayabilir.

Bu nedenle:

```python
max_depth
```

gibi hiperparametreler önemlidir.


In [ ]:
derinlik_sonuclari = []

for derinlik in range(1, 11):
    m = DecisionTreeClassifier(
        max_depth=derinlik,
        random_state=42
    )

    m.fit(
        X_train,
        y_train
    )

    derinlik_sonuclari.append({
        "Derinlik": derinlik,
        "TrainAccuracy": m.score(
            X_train,
            y_train
        ),
        "TestAccuracy": m.score(
            X_test,
            y_test
        )
    })

derinlik_df = pd.DataFrame(
    derinlik_sonuclari
)

derinlik_df


# 37. Train ve Test Accuracy Grafiği

In [ ]:
plt.plot(
    derinlik_df["Derinlik"],
    derinlik_df["TrainAccuracy"],
    marker="o",
    label="Train"
)

plt.plot(
    derinlik_df["Derinlik"],
    derinlik_df["TestAccuracy"],
    marker="o",
    label="Test"
)

plt.xlabel("Max Depth")
plt.ylabel("Accuracy")
plt.title("Decision Tree Karmaşıklığı")
plt.legend()
plt.grid()
plt.show()


Eğitim başarısı sürekli yükselirken test başarısının düşmeye başlaması overfitting konusunda önemli bir işaret olabilir.


# 38. Karar Ağacında Özellik Önemleri

Decision Tree hangi özellikleri bölmelerde daha fazla kullandığına ilişkin önem değerleri üretir.


In [ ]:
agac_onem = pd.DataFrame({
    "Ozellik": X.columns,
    "Onem": agac.feature_importances_
}).sort_values(
    "Onem",
    ascending=False
)

agac_onem


Özellik önemi nedensellik anlamına gelmez.

Modelin kendi karar yapısı içindeki kullanım bilgisidir.


# 39. Random Forest

Random Forest birçok karar ağacının sonuçlarını bir araya getiren bir ensemble yöntemidir.

Tek bir ağaca göre daha kararlı tahminler üretebilir.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

orman = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42
)

orman.fit(
    X_train,
    y_train
)

orman_pred = orman.predict(
    X_test
)

print(
    "Random Forest Accuracy:",
    accuracy_score(
        y_test,
        orman_pred
    )
)


# 40. Random Forest Özellik Önemleri

In [ ]:
orman_onem = pd.DataFrame({
    "Ozellik": X.columns,
    "Onem": orman.feature_importances_
}).sort_values(
    "Onem",
    ascending=False
)

orman_onem


# 41. Özellik Önemlerini Grafikleştirmek

In [ ]:
plt.bar(
    orman_onem["Ozellik"],
    orman_onem["Onem"]
)

plt.ylabel("Önem")
plt.title("Random Forest Özellik Önemleri")
plt.xticks(rotation=45)
plt.show()


# 42. Support Vector Machine

Support Vector Machine, sınıfları birbirinden ayıran karar sınırları oluşturmayı amaçlar.

Özellikle ölçeklendirme ile birlikte kullanılması önemlidir.


In [ ]:
from sklearn.svm import SVC

svm = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        SVC(
            kernel="rbf",
            probability=True,
            random_state=42
        )
    )
])

svm.fit(
    X_train,
    y_train
)

svm_pred = svm.predict(
    X_test
)

print(
    "SVM Accuracy:",
    accuracy_score(
        y_test,
        svm_pred
    )
)


`probability=True` kullandığımız için ileride `predict_proba()` ile sınıf olasılıklarını da görebileceğiz.


# 43. Gaussian Naive Bayes

Naive Bayes modelleri olasılıksal sınıflandırma yöntemleridir.

GaussianNB, özelliklerin sınıf içinde Gaussian dağılımla modellenebildiği bir Naive Bayes türüdür.


In [ ]:
from sklearn.naive_bayes import GaussianNB

naive_bayes = GaussianNB()

naive_bayes.fit(
    X_train,
    y_train
)

nb_pred = naive_bayes.predict(
    X_test
)

print(
    "GaussianNB Accuracy:",
    accuracy_score(
        y_test,
        nb_pred
    )
)


# 44. Bütün Modelleri Tek Fonksiyonla Değerlendirmek

Tekrar eden kodu azaltmak için bir değerlendirme fonksiyonu yazalım.


In [ ]:
def siniflandirma_metrikleri(
    model_adi,
    y_gercek,
    y_tahmin
):
    return {
        "Model": model_adi,
        "Accuracy": accuracy_score(
            y_gercek,
            y_tahmin
        ),
        "Precision": precision_score(
            y_gercek,
            y_tahmin,
            zero_division=0
        ),
        "Recall": recall_score(
            y_gercek,
            y_tahmin,
            zero_division=0
        ),
        "F1": f1_score(
            y_gercek,
            y_tahmin,
            zero_division=0
        )
    }


# 45. Model Sonuçlarını Toplamak

In [ ]:
sonuclar = [
    siniflandirma_metrikleri(
        "Dummy Baseline",
        y_test,
        baseline_pred
    ),
    siniflandirma_metrikleri(
        "Logistic Regression",
        y_test,
        lojistik_pred
    ),
    siniflandirma_metrikleri(
        "KNN",
        y_test,
        knn_pred
    ),
    siniflandirma_metrikleri(
        "Decision Tree",
        y_test,
        agac_pred
    ),
    siniflandirma_metrikleri(
        "Random Forest",
        y_test,
        orman_pred
    ),
    siniflandirma_metrikleri(
        "SVM",
        y_test,
        svm_pred
    ),
    siniflandirma_metrikleri(
        "GaussianNB",
        y_test,
        nb_pred
    )
]

model_df = pd.DataFrame(
    sonuclar
)

model_df


# 46. Accuracy'ye Göre Sıralamak

In [ ]:
model_df.sort_values(
    "Accuracy",
    ascending=False
)


# 47. F1-Score'a Göre Sıralamak

In [ ]:
model_df.sort_values(
    "F1",
    ascending=False
)


Accuracy'ye göre en iyi model ile F1-score'a göre en iyi model her zaman aynı olmak zorunda değildir.


# 48. Model Karşılaştırma Grafiği

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    model_df["Model"],
    model_df["F1"]
)

plt.ylabel("F1")
plt.title("Modellerin F1-Score Karşılaştırması")
plt.xticks(rotation=35)
plt.show()


# 49. Accuracy ve F1 Birlikte

In [ ]:
grafik_df = model_df.set_index(
    "Model"
)[["Accuracy", "F1"]]

grafik_df.plot(
    kind="bar",
    figsize=(11, 5)
)

plt.ylabel("Skor")
plt.title("Accuracy ve F1 Karşılaştırması")
plt.xticks(rotation=35)
plt.show()


# 50. En İyi Model Hangisi?

Tek bir cevap yoktur.

Model seçiminde:

- problemin amacı,
- yanlış pozitiflerin maliyeti,
- yanlış negatiflerin maliyeti,
- sınıf dengesi,
- tahmin hızı,
- eğitim süresi,
- açıklanabilirlik,
- model boyutu,
- bakım kolaylığı

gibi faktörler birlikte değerlendirilmelidir.


# 51. False Positive Neden Önemli Olabilir?

Örneğin model:

```text
Normal sistem → İncelenmeli
```

şeklinde yanlış pozitif üretirse gereksiz kontrol işlemleri yapılabilir.

Bazı problemlerde yanlış pozitif maliyetli olabilir.


# 52. False Negative Neden Önemli Olabilir?

Model:

```text
İncelenmeli sistem → Normal
```

şeklinde yanlış negatif üretirse gerçekten incelenmesi gereken örnek gözden kaçabilir.

Bazı problemlerde recall değerinin yüksek olması bu nedenle daha önemli olabilir.


# 53. Random Forest Confusion Matrix

In [ ]:
orman_cm = confusion_matrix(
    y_test,
    orman_pred
)

print(orman_cm)


# 54. Confusion Matrix Görseli

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay(
    confusion_matrix=orman_cm,
    display_labels=[
        "Normal",
        "İncelenmeli"
    ]
).plot()

plt.title("Random Forest Confusion Matrix")
plt.show()


# 55. Random Forest Classification Report

In [ ]:
print(
    classification_report(
        y_test,
        orman_pred,
        target_names=[
            "Normal",
            "İncelenmeli"
        ]
    )
)


# 56. Tahmin Olasılığı

Bazı sınıflandırma modelleri yalnızca sınıf değil sınıf olasılıkları da üretebilir.

Random Forest için:


In [ ]:
ilk_ornek = X_test.iloc[[0]]

print(
    orman.predict_proba(
        ilk_ornek
    )
)


# 57. Tek Örneğin Sınıf ve Olasılık Tahmini

In [ ]:
sinif = orman.predict(
    ilk_ornek
)[0]

olasiliklar = orman.predict_proba(
    ilk_ornek
)[0]

print(
    "Tahmin:",
    "İncelenmeli" if sinif == 1 else "Normal"
)

print(
    "Normal olasılığı:",
    round(
        olasiliklar[0],
        3
    )
)

print(
    "İncelenmeli olasılığı:",
    round(
        olasiliklar[1],
        3
    )
)


Tahmin olasılığı modelin sınıflara verdiği skordur.

Bunu gerçek dünyada kesinlik veya garanti olarak yorumlamamalıyız.


# 58. Yeni Veri İçin Tahmin

Sekiz sensör değerine sahip yeni bir örnek oluşturalım.


In [ ]:
yeni_ornek = pd.DataFrame(
    [[
        0.4,
        -1.1,
        1.2,
        0.8,
        -0.5,
        1.7,
        0.2,
        -0.9
    ]],
    columns=ozellik_adlari
)

yeni_ornek


# 59. Random Forest ile Yeni Tahmin

In [ ]:
yeni_tahmin = orman.predict(
    yeni_ornek
)[0]

print(
    "Tahmin edilen durum:",
    "İncelenmeli"
    if yeni_tahmin == 1
    else "Normal"
)


# 60. Tahmin Fonksiyonu Yazmak

In [ ]:
def durum_tahmin(
    model,
    sensor_degerleri
):
    veri = pd.DataFrame(
        [sensor_degerleri],
        columns=ozellik_adlari
    )

    sinif = model.predict(
        veri
    )[0]

    if hasattr(
        model,
        "predict_proba"
    ):
        olasilik = model.predict_proba(
            veri
        )[0]

        return {
            "sinif": int(sinif),
            "etiket": (
                "İncelenmeli"
                if sinif == 1
                else "Normal"
            ),
            "normal_olasilik": float(
                olasilik[0]
            ),
            "incelenmeli_olasilik": float(
                olasilik[1]
            )
        }

    return {
        "sinif": int(sinif),
        "etiket": (
            "İncelenmeli"
            if sinif == 1
            else "Normal"
        )
    }


In [ ]:
rapor = durum_tahmin(
    orman,
    [
        0.4,
        -1.1,
        1.2,
        0.8,
        -0.5,
        1.7,
        0.2,
        -0.9
    ]
)

print(rapor)


Bu yapı daha sonra Flask API'nin JSON cevabına dönüştürülebilir.


# 61. Logistic Regression Olasılıkları

In [ ]:
lojistik_olasilik = lojistik.predict_proba(
    ilk_ornek
)[0]

print(lojistik_olasilik)


# 62. SVM Olasılıkları

In [ ]:
svm_olasilik = svm.predict_proba(
    ilk_ornek
)[0]

print(svm_olasilik)


# 63. Farklı Modeller Aynı Örnek İçin Ne Diyor?

Aynı örneği bütün modellerle tahmin edelim.


In [ ]:
modeller = {
    "Logistic Regression": lojistik,
    "KNN": knn,
    "Decision Tree": agac,
    "Random Forest": orman,
    "SVM": svm,
    "GaussianNB": naive_bayes
}

for ad, m in modeller.items():
    tahmin = m.predict(
        ilk_ornek
    )[0]

    print(
        ad,
        "->",
        "İncelenmeli"
        if tahmin == 1
        else "Normal"
    )


Farklı algoritmalar aynı veriyi farklı karar mekanizmalarıyla değerlendirir.

Bu nedenle model seçimi yaparken sistematik değerlendirme gerekir.


# 64. Ölçeklendirmeyi Neden Pipeline İçinde Yaptık?

Yanlış yaklaşım:

```text
Bütün veriyi scale et
↓
Train-test ayır
```

Bu yaklaşım test verisindeki bilgilerin eğitim ön işlemesine karışmasına neden olabilir.

Pipeline:

```text
Train
↓
Scaler fit
↓
Model fit

Test
↓
Aynı scaler transform
↓
Model predict
```

akışını daha güvenli biçimde yönetir.


# 65. Hangi Modellerde Ölçeklendirme Daha Önemli?

Bu dersteki genel yaklaşım:

### Ölçeklendirmeden faydalanan modeller

- Logistic Regression
- KNN
- SVM

### Ağaç tabanlı modeller

- Decision Tree
- Random Forest

ağaçların bölme mantığı nedeniyle genellikle StandardScaler'a ihtiyaç duymaz.

Model davranışını algoritmanın çalışma mantığıyla ilişkilendirmek önemlidir.


# 66. Logistic Regression'ın Güçlü Yönleri

- iyi bir baseline olabilir,
- hızlıdır,
- olasılık üretir,
- doğrusal karar sınırlarında etkili olabilir,
- katsayıları yorumlamak mümkündür.

Ancak karmaşık doğrusal olmayan ilişkilerde tek başına yetersiz kalabilir.


# 67. KNN'nin Güçlü ve Zayıf Yönleri

Güçlü yönleri:

- mantığı anlaşılır,
- eğitim aşaması basittir,
- karmaşık karar sınırları oluşturabilir.

Dikkat edilmesi gerekenler:

- ölçeklendirme,
- K seçimi,
- çok büyük veri kümelerinde tahmin maliyeti,
- çok fazla özellikte uzaklık kavramının zorlaşması.


# 68. Decision Tree'nin Güçlü ve Zayıf Yönleri

Güçlü yönleri:

- açıklanabilir karar yapısı,
- doğrusal olmayan ilişkiler,
- ölçeklendirmeye ihtiyaç duymaması.

Dikkat edilmesi gereken:

- derin ağaçların overfitting yapabilmesi.


# 69. Random Forest'ın Güçlü ve Zayıf Yönleri

Güçlü yönleri:

- birçok problemde güçlü başlangıç modeli,
- doğrusal olmayan ilişkileri öğrenebilme,
- tek ağaca göre daha kararlı yapı,
- özellik önemleri sunabilmesi.

Dikkat edilmesi gerekenler:

- tek ağaca göre daha büyük ve karmaşık olması,
- tahmin ve eğitim maliyetinin artabilmesi,
- yorumlamanın tek ağaca göre zorlaşması.


# 70. SVM'nin Güçlü ve Zayıf Yönleri

Güçlü yönleri:

- güçlü karar sınırları,
- kernel ile doğrusal olmayan ilişkileri modelleyebilme,
- orta büyüklükte veri kümelerinde etkili olabilme.

Dikkat edilmesi gerekenler:

- ölçeklendirme,
- C ve kernel gibi hiperparametreler,
- çok büyük veri kümelerinde hesaplama maliyeti.


# 71. Naive Bayes'in Güçlü ve Zayıf Yönleri

Güçlü yönleri:

- hızlı,
- basit,
- olasılıksal,
- bazı metin sınıflandırma problemlerinde oldukça kullanışlı.

Dikkat edilmesi gereken:

- özellikler hakkında güçlü bağımsızlık varsayımları kullanması.

İleride metin sınıflandırma dersinde Naive Bayes'i yeniden kullanacağız.


# 72. Accuracy Paradoksu

Bir veri kümesinde:

```text
%95 Normal
%5 İncelenmeli
```

olsun.

Model her zaman:

```text
Normal
```

derse accuracy:

```text
%95
```

olabilir.

Ama model hiçbir pozitif örneği yakalayamaz.

Bu nedenle özellikle dengesiz sınıflarda:

- precision,
- recall,
- F1,
- confusion matrix

birlikte incelenmelidir.


# 73. Sınıf Dengesizliği

Bu derste sınıfları %65 / %35 civarında oluşturarak küçük bir dengesizlik kullandık.

Gerçek problemler çok daha dengesiz olabilir.

Örneğin:

```text
Normal işlem → 99.5%
Şüpheli işlem → 0.5%
```

Bu tür problemler için model değerlendirme ve örnekleme stratejileri ayrıca ele alınmalıdır.


# 74. Modeli Dosyaya Kaydetmek

En uygun olduğunu düşündüğümüz modeli dosyaya kaydedebiliriz.


In [ ]:
import joblib

joblib.dump(
    orman,
    "siniflandirma_modeli.joblib"
)

print("Model kaydedildi.")


# 75. Modeli Yeniden Yüklemek

In [ ]:
yuklenen_model = joblib.load(
    "siniflandirma_modeli.joblib"
)

print(
    yuklenen_model.predict(
        yeni_ornek
    )
)


# 76. Model + Tkinter

Masaüstü uygulaması:

```text
Sensör Değerleri
↓
Entry Alanları
↓
Tahmin Butonu
↓
Model.predict()
↓
Sonuç Label
```

şeklinde geliştirilebilir.


# 77. Model + Flask

Web uygulaması:

```text
HTML Form
↓
POST
↓
Flask Route
↓
joblib.load()
↓
model.predict()
↓
Jinja Sonuç Sayfası
```

akışıyla hazırlanabilir.


# 78. Model + API

JSON isteği:

```json
{
    "sensor_a": 0.4,
    "sensor_b": -1.1,
    "sensor_c": 1.2,
    "sensor_d": 0.8,
    "sensor_e": -0.5,
    "sensor_f": 1.7,
    "sensor_g": 0.2,
    "sensor_h": -0.9
}
```

cevap:

```json
{
    "sinif": 1,
    "etiket": "İncelenmeli"
}
```

şeklinde olabilir.

İlerleyen derslerde yapay zeka modellerini API olarak servis etmeyi uygulayacağız.


# 79. Model Karşılaştırma Süreci

Bu derste kullandığımız akış:

```text
Veri
↓
Train / Test
↓
Dummy Baseline
↓
Logistic Regression
↓
KNN
↓
Decision Tree
↓
Random Forest
↓
SVM
↓
Naive Bayes
↓
Accuracy / Precision / Recall / F1
↓
Confusion Matrix
↓
Model Karşılaştırma
↓
Yeni Veri Tahmini
```


# 80. Bir Sonraki Adım Neden Cross-Validation?

Tek bir train-test bölmesi şansa bağlı olarak bir modeli olduğundan iyi veya kötü gösterebilir.

Daha güvenilir değerlendirme için veriyi farklı eğitim-doğrulama bölmelerinde tekrar tekrar sınamak gerekir.

Bir sonraki derste:

- K-Fold,
- Stratified K-Fold,
- cross_val_score,
- cross_validate,
- GridSearchCV,
- RandomizedSearchCV,
- hiperparametre optimizasyonu

uygulayacağız.


# 81. Ders Özeti

Bu derste:

- ikili sınıflandırma,
- sentetik sınıflandırma verisi,
- sınıf dengesizliği,
- train-test,
- stratify,
- baseline,
- DummyClassifier,
- Logistic Regression,
- KNN,
- Decision Tree,
- Random Forest,
- SVM,
- Gaussian Naive Bayes,
- StandardScaler,
- Pipeline,
- accuracy,
- precision,
- recall,
- F1-score,
- confusion matrix,
- classification report,
- false positive,
- false negative,
- özellik önemi,
- `predict_proba()`,
- model karşılaştırma,
- model kaydetme,
- model yükleme

konularını öğrendik.


# 82. Mini Uygulamalar

1. `make_classification()` ile kendi ikili veri kümenizi oluşturun.
2. Sınıf dağılımını bulun.
3. Sınıf oranlarını yüzde olarak hesaplayın.
4. Veriyi stratified train-test olarak ayırın.
5. DummyClassifier oluşturun.
6. Baseline accuracy hesaplayın.
7. Logistic Regression Pipeline oluşturun.
8. KNN Pipeline oluşturun.
9. K=1 ile K=21 arasındaki tek değerleri karşılaştırın.
10. Decision Tree eğitin.
11. Farklı `max_depth` değerlerini karşılaştırın.
12. Random Forest eğitin.
13. Random Forest özellik önemlerini gösterin.
14. SVM modeli eğitin.
15. GaussianNB modeli eğitin.
16. Bütün modellerin accuracy değerlerini karşılaştırın.
17. Precision değerlerini karşılaştırın.
18. Recall değerlerini karşılaştırın.
19. F1-score değerlerini karşılaştırın.
20. En iyi üç modelin confusion matrix sonuçlarını inceleyin.
21. Classification report oluşturun.
22. Yeni bir örnek için tahmin üretin.
23. `predict_proba()` sonuçlarını inceleyin.
24. Seçtiğiniz modeli `joblib` ile kaydedin.
25. Kaydettiğiniz modeli yükleyip yeni veri için tekrar tahmin yapın.


# 83. Yapay Zeka Proje Görevi

Bir **Akıllı Durum Sınıflandırma Sistemi** geliştirin.

Konu örnekleri:

- sensör sistemi durumu,
- ürün kalite sınıflandırması,
- öğrenci çalışma durumu,
- üretim kontrolü,
- metin kategorisi,
- cihaz davranış sınıflandırması.

Projede en az:

- 500 örnek,
- en az 5 özellik,
- train-test ayrımı,
- baseline model,
- Logistic Regression,
- KNN,
- Decision Tree,
- Random Forest,
- en az bir ek sınıflandırıcı,
- accuracy,
- precision,
- recall,
- F1,
- confusion matrix,
- model karşılaştırma tablosu,
- yeni veri tahmini,
- model dosyasına kaydetme

bulunsun.

Ek geliştirme olarak:

- Tkinter arayüzü,
- Flask web formu,
- JSON API

özelliklerinden biri eklenebilir.


# Dersin Ana Kazanımı

Bu dersin sonunda öğrencinin artık tek bir algoritmaya bağlı kalmadan şu süreci kurabilmesi hedeflenmektedir:

**Sınıflandırma Problemi**

↓

**Veri Analizi**

↓

**Train / Test**

↓

**Baseline**

↓

**Birden Fazla Model**

↓

**Pipeline ve Ön İşleme**

↓

**fit() / predict()**

↓

**Accuracy**

↓

**Precision / Recall / F1**

↓

**Confusion Matrix**

↓

**Model Karşılaştırma**

↓

**Yeni Veri Tahmini**

↓

**Modeli Kaydet**

Bir sonraki derste bu karşılaştırmayı daha bilimsel hale getirerek **Cross Validation ve Hiperparametre Optimizasyonu** konularına geçeceğiz.
